# Курсовий проєкт. Ноутбук 3 — перенесення навчання й тонке налаштування

Другий підхід: замість навчання з нуля беруться мережі, попередньо навчені на
ImageNet, і донавчаються на картинах. Умова це прямо дозволяє — заборонені
лише набори даних із лабораторних, а не ваги ImageNet.

Одинадцять прогонів у чотирьох групах:

| Група | Прогони | Питання |
|---|---|---|
| бекбони | `b_frozen_*` | чиї готові ознаки краще розділяють художників |
| аугментація | `b4_strong_aug` | чи допоможе сильніша аугментація |
| глибина донавчання | `b5`–`b7` | скільки шарів розморожувати |
| дисбаланс | `b8`–`b10` | зважування класів і стрес-тест на всіх 50 класах |

Реєстр прогонів спільний із ноутбуком 2, тому в кінці порівняння «з нуля
проти перенесення навчання» збирається саме з виміряних чисел.

Потрібен ГП. Час: 1.5–2.5 години. Ноутбук можна зупиняти й запускати знову —
виконані прогони пропускаються.

Середовище, шляхи, зерно. Якщо список GPU порожній — Середовище виконання → Змінити тип → ГП L4, і перезапустити клітинку.

In [ ]:
# Environment, paths, seed. Same prelude in every notebook of the project.
import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if IN_COLAB and not tf.config.list_physical_devices('GPU'):
    print('\n!! ГП НЕ УВІМКНЕНО. Середовище виконання -> Змінити тип -> ГП L4.')

PROJECT_ROOT = Path('/content') if IN_COLAB else Path.cwd().parent
RESULTS = PROJECT_ROOT / 'results'
(RESULTS / 'raw').mkdir(parents=True, exist_ok=True)

# Everything that must survive a Colab disconnect lives on Drive.
DRIVE_DIR = (Path('/content/drive/MyDrive/dl_final_project') if IN_COLAB
             else PROJECT_ROOT / 'data' / 'drive')


def mount_drive():
    """Mount Drive once; locally just ensure the stand-in folder exists."""
    if IN_COLAB and not Path('/content/drive/MyDrive').exists():
        from google.colab import drive

        drive.mount('/content/drive')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    return DRIVE_DIR


SEED = 42
keras.utils.set_random_seed(SEED)
plt.rcParams['figure.dpi'] = 110
print('Проєкт:', PROJECT_ROOT)
print('Drive :', DRIVE_DIR)

## 1. Підготовлений набір

In [ ]:
# The prepared dataset comes from notebook 01: one tree with all 50 classes,
# already cleaned, split and resized to 320 px. Both tracks (top-15 and all-50)
# read from the same tree, so their splits are identical and comparable.
mount_drive()
PREPARED = DRIVE_DIR / 'prepared'
DATA_DIR = Path('/content/dataset') if IN_COLAB else PROJECT_ROOT / 'data' / 'dataset'

if not (DATA_DIR / 'train').exists():
    src = PREPARED / 'dataset.zip'
    if not src.exists():
        raise FileNotFoundError(
            f'немає {src} — спочатку виконайте ноутбук 01_dataset_eda.ipynb')
    t0 = time.time()
    with zipfile.ZipFile(src) as z:
        z.extractall(DATA_DIR)
    print(f'набір розпаковано за {time.time() - t0:.0f} с')

with open(PREPARED / 'class_names.json', encoding='utf-8') as f:
    CLASSES = json.load(f)
ALL50 = CLASSES['all50']
TOP15 = CLASSES['top15']

for _split in ('train', 'val', 'test'):
    _n = sum(1 for _ in (DATA_DIR / _split).rglob('*.jpg'))
    print(f'{_split:<6} {_n:>6} зображень')
print(f'\nкласів у наборі: {len(ALL50)}; основний трек: {len(TOP15)}')
print(', '.join(TOP15))

## 2. Конвеєр даних

In [ ]:
# tf.data pipeline (Keras 3 way; the labs used the legacy ImageDataGenerator).
# `class_names` restricts the reader to the chosen track without copying files.
#
# Only val/test are cached in memory: the training set at 224x224 float32 would
# take several GB, and decoding 320 px JPEGs with AUTOTUNE is cheap enough.
AUTOTUNE = tf.data.AUTOTUNE

AUG_PRESETS = {
    'light': [
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomRotation(0.05),
        keras.layers.RandomZoom(0.1),
    ],
    'strong': [
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomRotation(0.15),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomTranslation(0.1, 0.1),
        keras.layers.RandomContrast(0.2),
    ],
}


def make_augment(kind):
    """Augmentation as model layers: runs on GPU and is inactive at inference.

    No vertical flip: a painting is never upside down, and mirroring already
    stretches what a brushstroke may look like.
    """
    if not kind:
        return None
    return keras.Sequential(AUG_PRESETS[kind], name=f'augment_{kind}')


def make_datasets(class_names, img_size=224, batch=32):
    def load(split, shuffle):
        return keras.utils.image_dataset_from_directory(
            DATA_DIR / split, labels='inferred', label_mode='int',
            class_names=class_names, image_size=(img_size, img_size),
            batch_size=batch, shuffle=shuffle, seed=SEED,
            interpolation='bilinear', verbose=False)

    train = load('train', True).prefetch(AUTOTUNE)
    val = load('val', False).cache().prefetch(AUTOTUNE)
    test = load('test', False).cache().prefetch(AUTOTUNE)
    return train, val, test


def class_weights_for(class_names):
    """Balanced weights: rare artists get a proportionally larger penalty."""
    counts = np.array([len(list((DATA_DIR / 'train' / c).glob('*.jpg')))
                       for c in class_names], dtype=float)
    weights = counts.sum() / (len(counts) * counts)
    return {i: float(w) for i, w in enumerate(weights)}


def count_steps(class_names, split='train'):
    return sum(len(list((DATA_DIR / split / c).glob('*.jpg'))) for c in class_names)


print('зображень у навчальній вибірці трека top-15:', count_steps(TOP15))

## 3. Змішана точність

Обчислення у float16 при вагах у float32 і масштабуванні втрат. На L4 і A100 це
приблизно подвоює швидкість і не впливає на точність
([документація Keras](https://keras.io/api/mixed_precision/)). Саме завдяки їй
одинадцять експериментів укладаються в один сеанс.

In [ ]:
# Mixed precision: half-precision activations, float32 weights and loss scaling.
# Roughly doubles throughput on L4/A100 and costs nothing in accuracy, which is
# what makes a series of eleven experiments fit into one session.
# The last layer is forced back to float32 so softmax stays numerically stable.
if tf.config.list_physical_devices('GPU'):
    keras.mixed_precision.set_global_policy('mixed_float16')
print('політика точності:', keras.mixed_precision.global_policy().name)

## 4. Стенд прогонів

In [ ]:
# One function for every experiment, so that between any two runs exactly one
# factor differs. Each finished run is written to Drive and skipped on re-run,
# so a Colab disconnect costs one experiment instead of the whole series.
RUNS_DIR = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
CKPT_DIR = PROJECT_ROOT / 'ckpt'
for _d in (RUNS_DIR, MODELS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

FORCE_RERUN = False          # True -> ignore saved runs and train everything again
results = {}


class TimeHistory(keras.callbacks.Callback):
    """Wall-clock time of training; reported in the summary table."""

    def on_train_begin(self, logs=None):
        self._t0 = time.time()
        self.seconds = 0.0

    def on_train_end(self, logs=None):
        self.seconds = time.time() - self._t0


def make_callbacks(name, timer, patience=5, reduce_lr=True):
    # Checkpoints go to local disk: writing every improvement straight to Drive
    # is slow. The best model is copied to Drive once, after training.
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max',
                                      patience=patience, restore_best_weights=True,
                                      verbose=1),
        keras.callbacks.ModelCheckpoint(str(CKPT_DIR / f'{name}.keras'),
                                        monitor='val_accuracy', mode='max',
                                        save_best_only=True),
        keras.callbacks.CSVLogger(str(RESULTS / 'raw' / f'{name}_log.csv')),
        timer,
    ]
    # ReduceLROnPlateau and a LearningRateSchedule are mutually exclusive: the
    # callback assigns a new value to optimizer.learning_rate, and an optimizer
    # built with a schedule refuses that (TypeError, mid-training). A schedule
    # already governs the learning rate, so the callback is simply dropped.
    if reduce_lr:
        callbacks.insert(1, keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1))
    return callbacks


def top_k_accuracy(model, ds, k=3):
    hits = total = 0
    for xb, yb in ds:
        probs = model.predict(xb, verbose=0)
        topk = np.argsort(probs, axis=1)[:, -k:]
        hits += int(np.sum(topk == np.asarray(yb)[:, None]))
        total += len(yb)
    return hits / max(total, 1)


def run(cfg, build_fn):
    """Train one configuration and record everything the report needs."""
    name = cfg['name']
    path = RUNS_DIR / f'{name}.json'
    if path.exists() and not FORCE_RERUN:
        results[name] = json.loads(path.read_text(encoding='utf-8'))
        print(f"[{name}] уже виконано ранiше (val {results[name]['best_val_accuracy']:.4f})"
              ' — пропускаю')
        return results[name]

    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    classes = cfg.get('classes', TOP15)
    train, val, test = make_datasets(classes, cfg.get('img_size', 224),
                                     cfg.get('batch', 32))
    model = build_fn(cfg, classes)
    weights = class_weights_for(classes) if cfg.get('class_weight') else None

    timer = TimeHistory()
    print(f"\n{'=' * 70}\n{name}: {cfg['title']}\n{'=' * 70}")
    history = model.fit(train, validation_data=val, epochs=cfg.get('epochs', 25),
                        class_weight=weights, verbose=2,
                        callbacks=make_callbacks(name, timer,
                                                 cfg.get('patience', 5),
                                                 reduce_lr=not cfg.get('schedule')))

    test_loss, test_acc = model.evaluate(test, verbose=0)
    record = {
        'name': name,
        'title': cfg['title'],
        'config': {k: v for k, v in cfg.items() if k != 'classes'},
        'n_classes': len(classes),
        'params': int(model.count_params()),
        'trainable_params': int(sum(np.prod(w.shape) for w in model.trainable_weights)),
        'epochs_run': len(history.history['loss']),
        'best_val_accuracy': float(max(history.history['val_accuracy'])),
        'final_train_accuracy': float(history.history['accuracy'][-1]),
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'test_top3_accuracy': float(top_k_accuracy(model, test, 3)),
        'seconds': float(timer.seconds),
        'history': {k: [float(x) for x in v] for k, v in history.history.items()},
    }
    record['overfit_gap'] = record['final_train_accuracy'] - record['best_val_accuracy']

    model.save(MODELS_DIR / f'{name}.keras')
    path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
    results[name] = record
    print(f"\n[{name}] val {record['best_val_accuracy']:.4f} | "
          f"test {record['test_accuracy']:.4f} | top-3 {record['test_top3_accuracy']:.4f} | "
          f"{record['seconds'] / 60:.1f} хв")

    del model
    keras.backend.clear_session()
    return record


def load_previous_runs():
    """Pick up runs made in an earlier session (or in an earlier notebook)."""
    for f in sorted(RUNS_DIR.glob('*.json')):
        results.setdefault(f.stem, json.loads(f.read_text(encoding='utf-8')))
    print(f'відомо прогонів: {len(results)}')


load_previous_runs()

In [ ]:
# Summary table of every run known so far. pandas is a deliberate departure
# from the labs' hand-built ASCII tables: these numbers go straight into the report.
def summary_frame(names=None):
    names = names or list(results)
    rows = []
    for name in names:
        r = results[name]
        rows.append({
            'експеримент': name,
            'опис': r['title'],
            'класів': r['n_classes'],
            'параметрів': r['params'],
            'навчаних': r['trainable_params'],
            'епох': r['epochs_run'],
            'val acc': round(r['best_val_accuracy'], 4),
            'test acc': round(r['test_accuracy'], 4),
            'test top-3': round(r['test_top3_accuracy'], 4),
            'розрив': round(r['overfit_gap'], 4),
            'хв': round(r['seconds'] / 60, 1),
        })
    df = pd.DataFrame(rows).sort_values('test acc', ascending=False)
    return df.reset_index(drop=True)

In [ ]:
# Learning curves. Solid red = training, dashed blue = validation;
# the distance between them is overfitting.
def plot_histories(names, fname, suptitle):
    names = [n for n in names if results.get(n, {}).get('history')]
    if not names:
        print('немає історій для графіків')
        return
    cols = min(3, len(names))
    rows = -(-len(names) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 3.9 * rows),
                             squeeze=False)
    for ax, name in zip(axes.ravel(), names):
        rec = results[name]
        h = rec['history']
        epochs = range(1, len(h['accuracy']) + 1)
        # Маркери, а не самі лінії: прогін, зупинений після однієї епохи, інакше
        # не було б видно взагалі.
        ax.plot(epochs, h['accuracy'], color='crimson', marker='o', ms=3,
                label='навчальна')
        ax.plot(epochs, h['val_accuracy'], color='royalblue', ls='--',
                marker='s', ms=3, label='перевірочна')
        ax.axhline(rec['test_accuracy'], color='seagreen', ls=':', lw=1.2,
                   label='тестова')
        ax.set_title(f"{name}\n{rec['title']}", fontsize=9)
        ax.set_xlabel('епоха')
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_ylabel('точність')
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    for ax in axes.ravel()[len(names):]:
        ax.axis('off')
    fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS / fname, dpi=110, bbox_inches='tight')
    plt.show()

## 5. Побудова моделей

Схема однакова для всіх бекбонів: вхід → аугментація → приведення вхідного
діапазону → заморожена база → `GlobalAveragePooling2D` → `Dropout` → `Dense`
з `softmax`.

Два місця, де легко мовчки втратити точність:

**Вхідний діапазон.** У кожної мережі він свій. MobileNetV2 і ResNet50V2
очікують `[-1, 1]`, а EfficientNet тримає нормалізацію всередині себе й хоче
сирі `[0, 255]`. Якщо підставити звичне ділення на 255, помилки не буде — буде
просто гірший результат. Тут перетворення записане шаром `Rescaling`, а не
`Lambda` навколо `preprocess_input`: так збережений `.keras` відкривається в
ноутбуці 4 без `custom_objects` і без `safe_mode=False`.

**BatchNorm під час донавчання.** Шари нормалізації несуть статистики ImageNet.
Якщо дати міні-вибірці з 32 картин їх переписати, донавчання розвалиться. Тому
`unfreeze()` лишає всі `BatchNormalization` замороженими незалежно від глибини
розморожування.

In [ ]:
# Transfer learning: an ImageNet backbone with its own head replaced.
#
# Each network expects its own input range, and getting this wrong costs accuracy
# silently, without raising anything. MobileNetV2 and ResNet50V2 want [-1, 1];
# EfficientNet keeps its normalisation inside the model and must be fed raw
# [0, 255]. The conversion is expressed as a Rescaling layer rather than a Lambda
# wrapping preprocess_input, so the saved .keras file reloads in notebook 04
# without custom objects or safe_mode=False.
BACKBONES = {
    'MobileNetV2': keras.applications.MobileNetV2,
    'EfficientNetB0': keras.applications.EfficientNetB0,
    'ResNet50V2': keras.applications.ResNet50V2,
    'EfficientNetB3': keras.applications.EfficientNetB3,
}


def preprocess_layer(backbone):
    if backbone.startswith('EfficientNet'):
        return None
    return keras.layers.Rescaling(1. / 127.5, offset=-1, name='preprocess')


def make_optimizer(cfg, steps_per_epoch=None):
    lr = cfg.get('lr', 1e-3)
    if cfg.get('schedule') == 'cosine' and steps_per_epoch:
        lr = keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=lr,
            decay_steps=steps_per_epoch * cfg.get('epochs', 20))
    name = cfg.get('optimizer', 'adam')
    if name == 'adam':
        return keras.optimizers.Adam(learning_rate=lr)
    if name == 'rmsprop':
        return keras.optimizers.RMSprop(learning_rate=lr)
    if name == 'sgd':
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(f'невідомий оптимізатор: {name}')


def compile_model(model, cfg, classes):
    steps = -(-count_steps(classes) // cfg.get('batch', 32))
    model.compile(optimizer=make_optimizer(cfg, steps),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def find_backbone(model):
    """The nested pretrained model inside ours (the augment block is Sequential)."""
    for layer in model.layers:
        if isinstance(layer, keras.Model) and not isinstance(layer, keras.Sequential):
            return layer
    raise LookupError('у моделі немає вкладеного бекбона')


def unfreeze(base, fraction):
    """Unfreeze the last `fraction` of the backbone's layers.

    BatchNorm layers stay frozen regardless: they carry ImageNet running
    statistics, and letting mini-batches of 32 paintings rewrite them is the
    classic way to make fine-tuning diverge.
    """
    base.trainable = True
    cutoff = int(len(base.layers) * (1 - fraction))
    for layer in base.layers[:cutoff]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f'розморожено {trainable} з {len(base.layers)} шарів бекбона')


def build_transfer(cfg, classes):
    # Stage two: continue from a trained frozen model instead of starting over.
    if cfg.get('init_from'):
        src = MODELS_DIR / f"{cfg['init_from']}.keras"
        if not src.exists():
            raise FileNotFoundError(f'немає {src} — спочатку виконайте {cfg["init_from"]}')
        model = keras.models.load_model(src)
        unfreeze(find_backbone(model), cfg.get('unfreeze', 0.2))
        return compile_model(model, cfg, classes)

    size = cfg.get('img_size', 224)
    base = BACKBONES[cfg['backbone']](include_top=False, weights='imagenet',
                                      input_shape=(size, size, 3))
    base.trainable = False

    augment = make_augment(cfg.get('augment'))
    inputs = keras.Input(shape=(size, size, 3))
    x = augment(inputs) if augment is not None else inputs
    pre = preprocess_layer(cfg['backbone'])
    if pre is not None:
        x = pre(x)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(cfg.get('dropout', 0.3))(x)
    outputs = keras.layers.Dense(len(classes), activation='softmax',
                                 dtype='float32')(x)
    model = keras.Model(inputs, outputs, name=cfg['name'])
    return compile_model(model, cfg, classes)


def run_cfg(cfg):
    return run(cfg, build_transfer)


print('доступні бекбони:', ', '.join(BACKBONES))

## 6. Три бекбони із замороженою базою

In [ ]:
# B1-B3: the same head on three different ImageNet backbones, everything else
# identical. The base is frozen, so this measures how well each network's
# ready-made features already separate painters.
FROZEN_DEFAULTS = {'augment': 'light', 'lr': 1e-3, 'epochs': 20,
                   'dropout': 0.3, 'batch': 32, 'img_size': 224}

for backbone in ('MobileNetV2', 'EfficientNetB0', 'ResNet50V2'):
    run_cfg({**FROZEN_DEFAULTS,
             'name': f'b_frozen_{backbone.lower()}',
             'title': f'{backbone}, заморожена база',
             'backbone': backbone})

FROZEN_RUNS = [f'b_frozen_{b.lower()}'
               for b in ('MobileNetV2', 'EfficientNetB0', 'ResNet50V2')]
BEST_FROZEN = max(FROZEN_RUNS, key=lambda n: results[n]['best_val_accuracy'])
BEST_BACKBONE = results[BEST_FROZEN]['config']['backbone']
print(f'\nнайкращий бекбон: {BEST_BACKBONE} ({BEST_FROZEN})')
print(summary_frame(FROZEN_RUNS).to_string(index=False))

## 7. Сила аугментації

In [ ]:
# B4: the augmentation question, asked against the best frozen model so that
# strength of augmentation is the only thing that changes.
run_cfg({**FROZEN_DEFAULTS, 'backbone': BEST_BACKBONE, 'augment': 'strong',
         'name': 'b4_strong_aug',
         'title': f'{BEST_BACKBONE}, заморожена база, сильна аугментація'})

## 8. Глибина донавчання

Донавчання робиться у два етапи: спершу навчається лише голова на замороженій
базі (це вже зроблено вище), і лише потім розморожується частина бази з малою
швидкістю навчання. Порядок не декоративний: якщо розморозити базу, поки
випадково ініціалізована голова ще дає великі градієнти, перші ж кроки зіпсують
саме ті ознаки, заради яких бралася попередньо навчена мережа.

Усі три прогони стартують з **однієї й тієї самої** навченої замороженої
моделі, тож між ними відрізняється лише глибина. Криві нижче показують другий
етап.

In [ ]:
# B5-B7: how deep to fine-tune. All three continue from the same frozen model,
# so the depth of unfreezing is the only difference between them.
#
# A two-stage protocol is deliberate: unfreezing a backbone while the randomly
# initialised head still produces large gradients destroys the pretrained
# features in the first few steps. The curves below cover stage two only.
run_cfg({**FROZEN_DEFAULTS, 'backbone': BEST_BACKBONE,
         'name': 'b5_ft_20', 'title': 'донавчання останніх 20 % шарів, lr 1e-4',
         'init_from': BEST_FROZEN, 'unfreeze': 0.2, 'lr': 1e-4,
         'epochs': 15, 'patience': 4})

run_cfg({**FROZEN_DEFAULTS, 'backbone': BEST_BACKBONE,
         'name': 'b6_ft_50', 'title': 'донавчання останніх 50 % шарів, lr 5e-5',
         'init_from': BEST_FROZEN, 'unfreeze': 0.5, 'lr': 5e-5,
         'epochs': 15, 'patience': 4})

run_cfg({**FROZEN_DEFAULTS, 'backbone': BEST_BACKBONE,
         'name': 'b7_ft_all_cosine',
         'title': 'донавчання всієї бази, косинусний розклад від 1e-4',
         'init_from': BEST_FROZEN, 'unfreeze': 1.0, 'lr': 1e-4,
         'schedule': 'cosine', 'epochs': 15, 'patience': 5})

FT_RUNS = ['b5_ft_20', 'b6_ft_50', 'b7_ft_all_cosine']
BEST_FT = max(FT_RUNS, key=lambda n: results[n]['best_val_accuracy'])
print(f'\nнайкраща глибина донавчання: {BEST_FT}')
print(summary_frame([BEST_FROZEN] + FT_RUNS).to_string(index=False))

## 9. Дисбаланс класів

`class_weight` множить внесок кожного класу у втрати на величину, обернену до
його розміру — модель сильніше штрафується за помилки на рідкісних художниках.

Прогони `b9` і `b10` — це постановка на **всіх 50 класах**, включно з
художником, у якого всього кілька десятків картин. Саме тут загальна точність
перестає бути чесною мірою, і потрібні macro-метрики з ноутбука 4.

In [ ]:
# B8-B10: the imbalance questions.
#
# B8 repeats the best fine-tuning run with balanced class weights. B9 and B10 are
# the stress test the task asks for — all 50 artists, including the one with
# ~24 paintings, where the rarest classes are exactly what a plain accuracy
# number hides.
best_ft_cfg = dict(results[BEST_FT]['config'])

run_cfg({**best_ft_cfg, 'name': 'b8_class_weight',
         'title': f'{BEST_FT} + зважування класів', 'class_weight': True})

run_cfg({**FROZEN_DEFAULTS, 'backbone': BEST_BACKBONE, 'classes': ALL50,
         'name': 'b9_all50_frozen',
         'title': 'усі 50 класів, заморожена база, зважування класів',
         'class_weight': True, 'epochs': 25})

run_cfg({**FROZEN_DEFAULTS, 'backbone': BEST_BACKBONE, 'classes': ALL50,
         'name': 'b10_all50_ft',
         'title': 'усі 50 класів, донавчання, зважування класів',
         'init_from': 'b9_all50_frozen',
         'unfreeze': results[BEST_FT]['config'].get('unfreeze', 0.2),
         'lr': 5e-5, 'schedule': results[BEST_FT]['config'].get('schedule'),
         'class_weight': True, 'epochs': 15, 'patience': 4})

## 10. Додатковий експеримент

Більший бекбон на більшому вході. Вимкнений за замовчуванням: EfficientNetB3 на
300×300 коштує вчетверо більше за B0 на 224×224. Увімкніть `RUN_EXTRA = True`,
якщо лишився час.

In [ ]:
# B11: a bigger backbone at a bigger input, run only if there is time left.
# EfficientNetB3 at 300x300 is about four times the compute of B0 at 224.
RUN_EXTRA = False

if RUN_EXTRA:
    run_cfg({**FROZEN_DEFAULTS, 'backbone': 'EfficientNetB3', 'img_size': 300,
             'batch': 16, 'name': 'b11_effnetb3_300',
             'title': 'EfficientNetB3, вхід 300x300, заморожена база'})
else:
    print('b11 пропущено (RUN_EXTRA = False)')

## 11. Результати

In [ ]:
# Everything measured so far, including the runs from notebook 02 — they were
# written to the same registry on Drive, so the comparison between "from scratch"
# and "transfer learning" needs no copying of numbers by hand.
transfer_runs = [n for n in results if n.startswith('b')]
scratch_runs = [n for n in results if n.startswith('a')]

print('ПЕРЕНЕСЕННЯ НАВЧАННЯ')
print(summary_frame(transfer_runs).to_string(index=False))

if scratch_runs:
    print('\nМЕРЕЖІ З НУЛЯ (ноутбук 02)')
    print(summary_frame(scratch_runs).to_string(index=False))

    best_scratch = max(scratch_runs, key=lambda n: results[n]['test_accuracy'])
    best_transfer = max([n for n in transfer_runs if results[n]['n_classes'] == len(TOP15)],
                        key=lambda n: results[n]['test_accuracy'])
    gain = results[best_transfer]['test_accuracy'] - results[best_scratch]['test_accuracy']
    print(f'\nнайкраща з нуля    : {best_scratch} — '
          f"{results[best_scratch]['test_accuracy']:.4f}")
    print(f'найкраща з переносом: {best_transfer} — '
          f"{results[best_transfer]['test_accuracy']:.4f}")
    print(f'приріст: {gain:+.4f}')

print('\nдисбаланс: 15 класів проти 50')
for name in ('b9_all50_frozen', 'b10_all50_ft'):
    if name in results:
        r = results[name]
        print(f"  {name}: test {r['test_accuracy']:.4f}, "
              f"top-3 {r['test_top3_accuracy']:.4f} на {r['n_classes']} класах")

summary_frame(list(results)).to_csv(RESULTS / 'raw' / 'nb03_table.csv',
                                    index=False, encoding='utf-8')
BEST_OVERALL = summary_frame(
    [n for n in results if results[n]['n_classes'] == len(TOP15)]
).iloc[0]['експеримент']
(RESULTS / 'raw' / 'best_model.json').write_text(
    json.dumps({'best_top15': BEST_OVERALL,
                'best_all50': 'b10_all50_ft' if 'b10_all50_ft' in results else None},
               ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\nнайкраща модель проєкту: {BEST_OVERALL} — '
      f"test {results[BEST_OVERALL]['test_accuracy']:.4f}")

In [ ]:
plot_histories(transfer_runs, 'nb03_histories.png',
                'Криві навчання: перенесення навчання й донавчання')

## 12. Експорт

Зіп покласти у `project/results/incoming/`, ноутбук — у `project/notebooks/`.

In [ ]:
# Export: numbers for the report plus a zip to drop into results/incoming/.
NB = 'nb03'
payload = {
    'notebook': NB,
    'tensorflow': tf.__version__,
    'gpu': [d.name for d in tf.config.list_physical_devices('GPU')],
    'runs': results,
}
out = RESULTS / 'raw' / f'{NB}.json'
out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('записано', out)

zip_path = PROJECT_ROOT / f'project_{NB}_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESULTS.rglob('*')):
        if f.is_file() and f.suffix in ('.json', '.png', '.csv', '.md'):
            z.write(f, f.relative_to(RESULTS))
print('зіп:', zip_path, f'({zip_path.stat().st_size / 1e6:.1f} МБ)')

if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))